# MàthemaSharp.Quantum - API Reference

## Usage

Run after the Release build, on the .NET 11 kernel. The catalogue is extracted from the actual assemblies: exported types, constructors, methods, operators, properties and publicly declared fields. Accessors are represented by the properties. It includes defaults and generic constraints; members inherited from System.Object are not repeated. This notebook does not test every method: for verified examples use [consolidation](11-consolidation.ipynb).

## Reference



In [ ]:
#load "../shared/MathemaSharp.Numerics.csx"
using System;
using System.Linq;
using System.Reflection;
string TypeName(Type type)
{
    if (type.IsGenericParameter) return type.Name;
    if (type.IsArray) return TypeName(type.GetElementType()) + "[" + new string(',', type.GetArrayRank() - 1) + "]";
    if (type.IsByRef) return TypeName(type.GetElementType()) + "&";
    if (!type.IsGenericType) return type.FullName ?? type.Name;
    var name = type.GetGenericTypeDefinition().FullName;
    return name.Substring(0, name.IndexOf('`')) + "<" + string.Join(", ", type.GetGenericArguments().Select(TypeName)) + ">";
}
string Constraints(Type[] arguments)
{
    return string.Join(" ", arguments.Where(t => t.IsGenericParameter).Select(t =>
    {
        var parts = t.GetGenericParameterConstraints().Select(TypeName).ToList();
        var attrs = t.GenericParameterAttributes;
        if ((attrs & GenericParameterAttributes.ReferenceTypeConstraint) != 0) parts.Insert(0, "class");
        if ((attrs & GenericParameterAttributes.NotNullableValueTypeConstraint) != 0) parts.Insert(0, "struct");
        if ((attrs & GenericParameterAttributes.DefaultConstructorConstraint) != 0) parts.Add("new()");
        return parts.Count == 0 ? "" : "where " + t.Name + " : " + string.Join(", ", parts);
    }).Where(s => s.Length != 0));
}
string Parameters(ParameterInfo[] parameters) => string.Join(", ", parameters.Select(p =>
    TypeName(p.ParameterType) + " " + p.Name + (p.IsOptional ? " = " + (p.DefaultValue ?? "null") : "")));
var assembly = Assembly.Load("MathemaSharp.Quantum");
var exported = assembly.GetExportedTypes().OrderBy(t => t.FullName).ToArray();
Console.WriteLine("Assembly: " + assembly.GetName().Name);
Console.WriteLine("Runtime: " + Environment.Version);
Console.WriteLine("Exported types: " + exported.Length);
foreach (var type in exported)
{
    Console.WriteLine("\nTYPE " + TypeName(type) + " " + Constraints(type.GetGenericArguments()));
    var flags = BindingFlags.Public | BindingFlags.Instance | BindingFlags.Static | BindingFlags.DeclaredOnly;
    foreach (var constructor in type.GetConstructors(flags).OrderBy(c => c.ToString()))
        Console.WriteLine("  new " + TypeName(type) + "(" + Parameters(constructor.GetParameters()) + ")");
    foreach (var property in type.GetProperties(flags).OrderBy(p => p.Name))
        Console.WriteLine("  " + TypeName(property.PropertyType) + " " + property.Name + " { " + (property.GetMethod?.IsPublic == true ? "get; " : "") + (property.SetMethod?.IsPublic == true ? "set; " : "") + "}");
    foreach (var field in type.GetFields(flags).OrderBy(f => f.Name))
        Console.WriteLine("  field " + TypeName(field.FieldType) + " " + field.Name);
    foreach (var method in type.GetMethods(flags).Where(m => !m.IsSpecialName || m.Name.StartsWith("op_")).OrderBy(m => m.Name).ThenBy(m => m.ToString()))
    {
        var generic = method.IsGenericMethod ? "<" + string.Join(", ", method.GetGenericArguments().Select(t => t.Name)) + ">" : "";
        Console.WriteLine("  " + (method.IsStatic ? "static " : "") + TypeName(method.ReturnType) + " " + method.Name + generic + "(" + Parameters(method.GetParameters()) + ") " + Constraints(method.GetGenericArguments()));
    }
}
if (exported.Length == 0) throw new Exception("Empty API catalog");
Console.WriteLine("catalog: OK");


Assembly: MathemaSharp.Quantum
Runtime: 11.0.0
Exported types: 4

TYPE MathemaSharp.Quantum.ComplexMatrix<T> where T : System.Numerics.IFloatingPointIeee754<T>, System.Numerics.IMinMaxValue<T>
  new MathemaSharp.Quantum.ComplexMatrix<T>(System.Int32 rows, System.Int32 cols)
  new MathemaSharp.Quantum.ComplexMatrix<T>(System.Numerics.Complex<T>[,] values)
  System.Int32 Cols { get; }
  T DefaultTolerance { get; }
  System.Numerics.Complex<T> Item { get; set; }
  System.Int32 Rows { get; }
  MathemaSharp.Quantum.ComplexMatrix<T> Dagger() 
  static MathemaSharp.Quantum.ComplexMatrix<T> Identity(System.Int32 n) 
  System.Boolean IsHermitian() 
  System.Boolean IsHermitian(T tolerance) 
  System.Boolean IsUnitary() 
  System.Boolean IsUnitary(T tolerance) 
  static MathemaSharp.Quantum.ComplexMatrix<T> op_Addition(MathemaSharp.Quantum.ComplexMatrix<T> a, MathemaSharp.Quantum.ComplexMatrix<T> b) 
  static MathemaSharp.Quantum.ComplexMatrix<T> op_Multiply(MathemaSharp.Quantum.ComplexMatrix<T>

## Limits of the catalogue

Reflection also includes members generated for records (Equals, Deconstruct, etc.). It does not faithfully represent all nullable annotations, readonly/ref and the names of tuple elements: for these details consult the source and the build's XML files. The catalogue does not replace numerical contracts, tests or per-type coverage analysis. [Inventory and limits](../../ARCHITECTURE.md), [Usage](../../USAGE.md), [Roadmap](../../ROADMAP.md).

